# UFC Predictor — Model Training and Comparison

This notebook trains and compares Random Forest and CatBoost using a chronological split. Log loss is the primary selection metric because the predictor is intended to produce reliable win probabilities.

- **Train:** before 2022
- **Validation:** 2022–2023
- **Test:** 2024 onward


## 1. Setup

Locate the project and import the shared modeling utilities.


In [1]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
candidates = [current, *current.parents]
project_root = next(
    candidate
    for candidate in candidates
    if (candidate / "data/processed/ufcstats/fight_dataset.parquet").exists()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

WindowsPath('C:/Users/pablo/Desktop/PABLO/PROGRAMACION/UFC_PREDICTOR')

In [2]:
from src.modeling.ufc_modeling import (
    CatBoostConfig,
    RandomForestConfig,
    SplitConfig,
    add_context_features,
    build_feature_frame,
    build_random_forest_pipeline,
    catboost_feature_importance,
    load_dataset,
    metrics_from_probabilities,
    random_forest_feature_importance,
    temporal_split,
    train_and_compare,
    train_catboost,
)

import pandas as pd

## 2. Dataset Validation

Check the dataset size, date range, and target balance before training.


In [3]:
df = load_dataset(project_root)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Earliest date:", df["event_date"].min())
print("Latest date:", df["event_date"].max())
print("Mean target:", df["target"].mean())

df.head()

Rows: 8628
Columns: 126
Earliest date: 1994-03-11 00:00:00
Latest date: 2026-07-18 00:00:00
Mean target: 0.500579508576727


,fight_id,event_id,event_name,event_date,fight_order,weight_class,fighter_a_id,fighter_a_name,fighter_b_id,fighter_b_name,...,diff_takedown_defense_before,diff_control_seconds_before,diff_knockdowns_before,diff_knockdowns_absorbed_before,diff_submission_attempts_before,diff_reversals_before,diff_height_cm,diff_reach_cm,diff_age_years,target
0,4acab67848e78327,a6a9ab5a824e8f66,UFC 2: No Way Out,1994-03-11,15,Open Weight,a683f9ddb70aa4bd,Sean Daugherty,be9d259be012e8a4,Scott Morris,...,<NA>,0.0,0.0,0.0,0.0,0.0,5.08,<NA>,<NA>,0
1,4b9ae533ccb3fcdf,a6a9ab5a824e8f66,UFC 2: No Way Out,1994-03-11,14,Open Weight,46c8ec317aff28ac,Patrick Smith,ea0ad155451ed1f5,Ray Wizard,...,<NA>,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,1
2,ccee020be2e8d9f6,a6a9ab5a824e8f66,UFC 2: No Way Out,1994-03-11,13,Open Weight,319fa1bd3176bded,Johnny Rhodes,49590e0508b2c19f,David Levicki,...,<NA>,0.0,0.0,0.0,0.0,0.0,-12.7,<NA>,<NA>,1
3,d917c8c7461b9831,a6a9ab5a824e8f66,UFC 2: No Way Out,1994-03-11,12,Open Weight,505934897b8b4824,Thaddeus Luster,c3c23c99477c041b,Frank Hamaker,...,<NA>,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,0
4,3b020d4914b44fc8,a6a9ab5a824e8f66,UFC 2: No Way Out,1994-03-11,11,Open Weight,96087e90d900f0ef,Robert Lucarelli,a6c2f5381d575920,Orlando Wiet,...,<NA>,0.0,0.0,0.0,0.0,0.0,10.16,<NA>,<NA>,0


## 3. Feature Construction

Build the numerical and categorical feature sets while excluding identifiers and post-fight information to prevent leakage.


In [4]:
X, numeric_features, categorical_features = build_feature_frame(df)
y = df["target"]

print("Numerical features:", len(numeric_features))
print("Categorical features:", categorical_features)
print("X shape:", X.shape)

X.head()

Numerical features: 47
Categorical features: ['weight_class', 'stance_matchup', 'same_stance']
X shape: (8628, 50)


,diff_age_years,diff_control_seconds_before,diff_control_stats_coverage_before,diff_current_loss_streak_before,diff_current_win_streak_before,diff_days_since_last_fight,diff_draws_before,diff_fights_before,diff_fights_with_control_stats_before,diff_fights_with_stats_before,...,total_fights_with_stats_before,min_fights_with_stats_before,total_stats_fight_time_seconds_before,min_stats_coverage_before,min_control_stats_coverage_before,both_debutants,one_debutant,weight_class,stance_matchup,same_stance
0,NaN,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,1.0,0.0,Open Weight,Unknown_vs_Orthodox,False
1,NaN,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,1.0,0.0,Open Weight,Orthodox_vs_Unknown,False
2,NaN,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,1.0,0.0,Open Weight,Orthodox_vs_Unknown,False
3,NaN,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,1.0,0.0,Open Weight,Unknown_vs_Unknown,False
4,NaN,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,1.0,0.0,Open Weight,Unknown_vs_Southpaw,False


## 4. Temporal Split

Split the data chronologically so model selection reflects performance on later fights.


In [5]:
split_config = SplitConfig(
    train_end="2022-01-01",
    validation_end="2024-01-01",
)

splits = temporal_split(df, split_config)

summary = pd.DataFrame(
    {
        "rows": {
            name: len(index)
            for name, index in splits.items()
        },
        "date_min": {
            name: df.loc[index, "event_date"].min()
            for name, index in splits.items()
        },
        "date_max": {
            name: df.loc[index, "event_date"].max()
            for name, index in splits.items()
        },
        "target_rate": {
            name: y.loc[index].mean()
            for name, index in splits.items()
        },
    }
)
summary

,rows,date_min,date_max,target_rate
train,6307,1994-03-11,2021-12-18,0.508166
validation,1010,2022-01-15,2023-12-16,0.481188
test,1311,2024-01-13,2026-07-18,0.479024


## 5. Model Training

Train both models, select the best validation log loss, and evaluate the resulting models on the held-out test period.


In [6]:
rf_config = RandomForestConfig(
    n_estimators=800,
    max_depth=12,
    min_samples_leaf=4,
)

cat_config = CatBoostConfig(
    iterations=2000,
    learning_rate=0.03,
    depth=5,
    l2_leaf_reg=10.0,
    early_stopping_rounds=150,
    verbose=100,
)

results = train_and_compare(
    project_root=project_root,
    rf_config=rf_config,
    cat_config=cat_config,
    split_config=split_config,
)

Training Random Forest...
Training CatBoost...
0:	learn: 0.6919777	test: 0.6921598	best: 0.6921598 (0)	total: 174ms	remaining: 5m 48s
100:	learn: 0.6436785	test: 0.6693868	best: 0.6693868 (100)	total: 4.86s	remaining: 1m 31s
200:	learn: 0.6269106	test: 0.6643399	best: 0.6643151 (198)	total: 9.56s	remaining: 1m 25s
300:	learn: 0.6131275	test: 0.6635690	best: 0.6632978 (288)	total: 14.3s	remaining: 1m 20s
400:	learn: 0.5961288	test: 0.6627661	best: 0.6627661 (400)	total: 18.8s	remaining: 1m 15s
500:	learn: 0.5796968	test: 0.6623928	best: 0.6620925 (493)	total: 23.4s	remaining: 1m 10s
600:	learn: 0.5633403	test: 0.6636993	best: 0.6619903 (538)	total: 28s	remaining: 1m 5s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.66199032
bestIteration = 538

Shrink model to first 539 iterations.

Training complete.
Selected model: catboost

     split         model  accuracy  balanced_accuracy  roc_auc  log_loss  brier_score  threshold  tn  fp  fn  tp
validation random_forest  0

## 6. Metric Comparison

Compare discrimination, classification, and probability-quality metrics across both splits.


In [7]:
results["metrics_dataframe"].sort_values(
    ["split", "log_loss"]
)

,split,model,accuracy,balanced_accuracy,roc_auc,log_loss,brier_score,threshold,tn,fp,fn,tp
3,test,catboost,0.629291,0.630368,0.675507,0.646044,0.227393,0.5,413,270,216,412
2,test,random_forest,0.610984,0.612157,0.664446,0.655810,0.231798,0.5,399,284,226,402
1,validation,catboost,0.593069,0.594395,0.639753,0.661990,0.234977,0.5,293,231,180,306
0,validation,random_forest,0.599010,0.600419,0.645981,0.662699,0.235146,0.5,295,229,176,310


In [8]:
print("Selected model:", results["selected_model"])
print("Artifacts:", results["artifact_directories"]["root"])

Selected model: catboost
Artifacts: C:\Users\pablo\Desktop\PABLO\PROGRAMACION\UFC_PREDICTOR\artifacts\modeling


## 7. Feature Importance

Inspect the features that contribute most strongly to each model.


In [9]:
rf_importance = random_forest_feature_importance(
    results["random_forest"]
)
rf_importance.head(20)

,feature,importance
0,numeric__diff_age_years,0.072080
1,numeric__diff_sig_str_absorbed_per_min_before,0.039213
2,numeric__diff_sig_str_defense_before,0.038328
3,numeric__diff_control_seconds_before,0.032641
4,numeric__diff_sig_str_landed_before,0.032468
5,numeric__diff_sig_str_absorbed_before,0.031513
6,numeric__diff_sig_str_absorbed_attempted_before,0.030941
7,numeric__diff_sig_str_attempted_before,0.030692
8,numeric__diff_sig_str_landed_per_min_before,0.030310
9,numeric__diff_takedowns_landed_before,0.029667


In [10]:
cat_importance = catboost_feature_importance(
    results["catboost"],
    list(results["features"].columns),
)
cat_importance.head(20)

,feature,importance
0,diff_age_years,9.881521
1,diff_reach_cm,7.559111
2,diff_sig_str_absorbed_per_min_before,4.823915
3,diff_takedowns_landed_before,4.055600
4,diff_sig_str_landed_before,3.908302
5,diff_sig_str_defense_before,3.659260
6,diff_takedowns_attempted_before,3.655929
7,diff_win_rate_before,3.437843
8,stance_matchup,3.236174
9,diff_sig_str_accuracy_before,3.086540


## 8. Test Predictions

Review the highest predicted probabilities as a final sanity check.


In [11]:
predictions_path = (
    results["artifact_directories"]["predictions"]
    / "test_predictions.parquet"
)

test_predictions = pd.read_parquet(predictions_path)
test_predictions.sort_values(
    "probability_fighter_a",
    ascending=False,
).head(20)

,fight_id,event_date,event_name,weight_class,fighter_a_id,fighter_a_name,fighter_b_id,fighter_b_name,target,model,probability_fighter_a,probability_fighter_b,prediction,predicted_winner_name,actual_winner_name,correct
1783,b35e47f2f58ef026,2024-11-16,UFC 309: Jones vs. Miocic,Heavyweight,07f72a2a7591b409,Jon Jones,d28dee5c705991df,Stipe Miocic,1,catboost,0.872339,0.127661,1,Jon Jones,Jon Jones,True
1850,daef1691c7d6b1e4,2025-01-18,UFC 311: Makhachev vs. Moicano,Lightweight,275aca31f61ba28c,Islam Makhachev,b6452706b373eea1,Renato Moicano,1,catboost,0.850932,0.149068,1,Islam Makhachev,Islam Makhachev,True
2388,8d740c844353ae0e,2026-02-21,UFC Fight Night: Strickland vs. Hernandez,Middleweight,093e1f5bb73850be,Anthony Hernandez,0d8011111be000b2,Sean Strickland,0,catboost,0.841819,0.158181,1,Anthony Hernandez,Sean Strickland,False
1782,4f4189009a190e35,2024-11-16,UFC 309: Jones vs. Miocic,Lightweight,07225ba28ae309b6,Charles Oliveira,4b93a88f3b1de35b,Michael Chandler,1,catboost,0.841044,0.158956,1,Charles Oliveira,Charles Oliveira,True
1723,998bde5639c61adb,2024-10-12,UFC Fight Night: Royval vs. Taira,Flyweight,4461d7e47375a895,Tatsuro Taira,6e15f63b6c2e2c15,Brandon Royval,0,catboost,0.802402,0.197598,1,Tatsuro Taira,Brandon Royval,False
1492,2db0998b24c11ea0,2024-05-04,UFC 301: Pantoja vs. Erceg,Middleweight,4126a78111c0855a,Caio Borralho,eabf206b162b3b83,Paul Craig,1,catboost,0.796983,0.203017,1,Caio Borralho,Caio Borralho,True
1842,aa3fca085d476636,2025-01-18,UFC 311: Makhachev vs. Moicano,Lightweight,99bd51917728c25d,Grant Dawson,f53d4f21d1b5f2dc,Diego Ferreira,1,catboost,0.796390,0.203610,1,Grant Dawson,Grant Dawson,True
1918,1c62a7444808d97f,2025-03-15,UFC Fight Night: Vettori vs. Dolidze 2,Women's Bantamweight,68d35296f566792b,Josiane Nunes,b1e5bcaad32a3cac,Priscila Cachoeira,0,catboost,0.796150,0.203850,1,Josiane Nunes,Priscila Cachoeira,False
2581,afc021700481ac4e,2026-06-20,UFC Fight Night: Kape vs. Horiguchi,Light Heavyweight,b4b496ec2197ee3e,Navajo Stirling,cd13728ae1151f46,Ion Cutelaba,1,catboost,0.794574,0.205426,1,Navajo Stirling,Navajo Stirling,True
2082,f722b96ae6862843,2025-06-28,UFC 317: Topuria vs. Oliveira,Flyweight,17e97649403ba428,Joshua Van,6e15f63b6c2e2c15,Brandon Royval,1,catboost,0.793577,0.206423,1,Joshua Van,Joshua Van,True


## Summary

CatBoost achieved the lowest validation log loss and was selected in this baseline comparison. The trained models, metrics, predictions, plots, and feature importances were saved for subsequent tuning and calibration.
